# DhakaFlow Nexus — MASTER NOTEBOOK
**The whole project, one notebook, six stages.** Run top to bottom.

GOAL: observe real traffic → predict congestion before it happens → reduce it
by best routing + signal control → test every decision inside a SUMO digital
twin of real Dhaka roads → prove total delay is reduced.

Each stage below runs the REAL component and shows its REAL result.

In [ ]:
# Stage 0 · Setup — everything runs from the repo root
import json, subprocess, sys
from pathlib import Path
REPO = Path.cwd() if (Path.cwd()/'PROJECT_MAP.md').exists() else Path.cwd().parent
%cd {REPO}
print('repo:', REPO)

## Stage 1 · OBSERVE — the Dhaka-trained eyes
Our RT-DETR r50, fine-tuned on 9,058 real Bangladeshi images.
Test result: **mAP50:95 0.711** (the dataset paper's best published: 0.643) —
rickshaw AP50 0.904 · CNG 0.961 · easy-bike 0.965.

In [ ]:
print(json.dumps(json.load(open('artifacts/p1_triage/triage.json'))['arms']['p1-rtdetr-r50']['per_class'], indent=1))

### Run it on ANY video (the presentation command)
Uncomment and point at any traffic video — the examiner's included.

In [ ]:
# !python scripts/case_report.py --source <ANY_VIDEO.mp4> --location farmgate --name live_demo
# then open artifacts/case_live_demo/case_report.html
from IPython.display import Image, display
display(Image('docs/media/react_frame_2.jpg'))   # real frame from the rush-hour run
print(json.dumps(json.load(open('artifacts/case_rush_test/case.json')), indent=1)[:600])

## Stage 2 · PREDICT — graph forecasting (official BasicTS harness)
Graph WaveNet predicts each of 250 sensors 15 min ahead:
**MAE 0.44 m/s — 3.2× better than the persistence floor (1.39).**

In [ ]:
for line in open('artifacts/p3_results/GWNet_cpu.log'):
    if 'Evaluate best model on test data for horizon 3,' in line: last = line
print(last.strip())

## Stage 3 · BEST ROUTE — Dijkstra on the real 6,527-edge network
Jam 10 edges → the router diverts and reports seconds saved. Computed live:

In [ ]:
from research.routing.r1_router import R1Router
r = R1Router()
free = r.best_route((23.7573, 90.3854), (23.7389, 90.3958))
jam = {e: 5.0 for e in r._dijkstra(free['src_edge'], free['dst_edge'], {})[2:12]}
print(json.dumps(r.best_route((23.7573, 90.3854), (23.7389, 90.3958), jam), indent=1))

## Stage 4 · SIGNALS — baselines + learned control (paired seeds 11-13)
fixed-time **535.9 s** · MAPPO+pressure 551.3 · MAPPO 558.9 · classical
pressure baseline 586.7 (+9.5% WORSE — local greed harms networks; the
motivation for coordinated learning). Reported honestly at-checkpoint.

In [ ]:
for arm in ('fixed_time','max_pressure'):
    d = json.load(open(f'artifacts/control_baselines/summary_{arm}.json'))
    print(f"{arm:14s} {d['mean_time_loss_s']:.1f} s")
for arm in ('mappo','mappo_pressure'):
    d = json.load(open(f'artifacts/rl_eval/eval_{arm}.json'))
    print(f"{arm:14s} {d['mean_time_loss_s']:.1f} s")

## Stage 5 · TEST IN SUMO — the same-state counterfactual (THE PROOF)
One simulation runs until real congestion (t0). `traci.saveState` freezes the
exact state. Branch A does nothing; branch B acts (green discharge +
capacity-gated bounded rerouting). Identical pasts ⇒ the difference is CAUSED
by the control. Result: **target queue −75% at ~neutral network cost.**

In [ ]:
print(json.dumps(json.load(open('artifacts/same_state/kb_mild_gated/result.json'))['four_pane'], indent=1))
# rerun live (≈10 min):  !python scripts/same_state_counterfactual.py --camera karwanbazar --seed 11

## Stage 6 · PROVE — network-scale statistics
Offline network experiment (separate from any single video): system-optimal
vs selfish routing, 10 seeds, identical demand:
**−19.5% mean delay, p = 1.3×10⁻⁵, Cohen's d 2.98.**

In [ ]:
d = json.load(open('artifacts/due_dso_stats/v200/result.json'))
print(json.dumps({c['metric']: c['dso_improvement_percent'] for c in d['comparisons']}, indent=1))

## The four-pane demo (what sir sees)
`!python scripts/four_pane_report.py --case rush_test --counterfactual kb_mild_gated`
→ open `artifacts/four_pane.html`

## Where everything lives
`PROJECT_MAP.md` (6 main files) · `docs/SYSTEM_DESIGN.md` (architecture +
literature) · `docs/RESULTS_MATRIX.md` (all numbers) ·
`docs/PRESENTER_GUIDE.md` (what to say)